In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token="KqFKvQsgtV6p3DlT6sJeV0qg3_ZlatQZi6sPSAMJd8IN",
    overwrite=True
)

In [4]:
from qiskit_ibm_runtime import QiskitRuntimeService


# Les fois suivantes : charger
service = QiskitRuntimeService()


InvalidAccountError: 'Unable to retrieve instances. Please check that you are using a valid API token.'

In [5]:
# Voir tous les backends (ordis quantiques sur le Cloud)
backends = service.backends()

print("Backends disponibles :")

for backend in backends:
    status = backend.status()
    print(f"\n{backend.name}")
    print(f"  Qubits: {backend.num_qubits}")
    print(f"  En attente: {status.pending_jobs}")
    print(f"  Opérationnel: {status.operational}")

# Filtrer les simulateurs
simulators = service.backends(
    simulator=True
)
print(simulators)
# Filtrer les vrais ordinateurs
real_backends = service.backends(
    simulator=False,
    operational=True
)
print(real_backends)

qiskit_runtime_service.backends:WARNING:2026-06-16 15:06:34,458: Loading instance: test, plan: open


Backends disponibles :

ibm_fez
  Qubits: 156
  En attente: 8
  Opérationnel: True

ibm_marrakesh
  Qubits: 156
  En attente: 7
  Opérationnel: True

ibm_kingston


qiskit_runtime_service.backends:WARNING:2026-06-16 15:06:39,289: Loading instance: test, plan: open
qiskit_runtime_service.backends:WARNING:2026-06-16 15:06:39,295: Loading instance: test, plan: open


  Qubits: 156
  En attente: 89
  Opérationnel: True
[]
[<IBMBackend('ibm_fez')>, <IBMBackend('ibm_marrakesh')>, <IBMBackend('ibm_kingston')>]


In [6]:
from qiskit_ibm_runtime import (
    QiskitRuntimeService)

from qiskit import QuantumCircuit
# from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# Se connecter
service = QiskitRuntimeService()

# Choisir le backend le moins occupé
backend = service.least_busy(
    operational=True,
    simulator=False,
    min_num_qubits=50
)

print(f"Backend sélectionné: {backend.name}")
print(f"Nombre de qubits: {backend.num_qubits}")

qiskit_runtime_service.__init__:WARNING:2026-06-16 15:06:44,571: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: test. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-06-16 15:06:44,915: Loading instance: test, plan: open
qiskit_runtime_service.backends:WARNING:2026-06-16 15:06:48,469: Using instance: test, plan: open


Backend sélectionné: ibm_marrakesh
Nombre de qubits: 156


In [7]:
# Créer un circuit simple
qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.measure([0, 1], [0, 1])

print(qc.draw())

     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 


In [8]:
# Transpiler pour le backend
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# On lance la transpilation (équivalent de la compilation)
pm = generate_preset_pass_manager(
    backend=backend,
    optimization_level=3
)
# Voici le circuit transpilé
transpiled_qc = pm.run(qc)

# Créer le job
from qiskit_ibm_runtime import (SamplerV2 as Sampler)
sampler = Sampler(backend)
job = sampler.run([transpiled_qc], shots=1024)

print(f"Job ID: {job.job_id()}")
print("Job soumis ! En attente...")

# Attendre et récupérer les résultats
result = job.result()

# Extraire les comptages
pub_result = result[0]
counts = pub_result.data.c.get_counts()

print("\nRésultats :")
print(counts)

Job ID: d8okkse8aqlc73egmak0
Job soumis ! En attente...

Résultats :
{'00': 524, '11': 466, '10': 27, '01': 7}
